# GarmentLab · Controlled regularization study

This notebook records an eight-candidate MobileNetV3-Small study with discovery, fresh-seed confirmation and paired final fits. The completed experiment contains 20 fits of 18 epochs each.

Temperature fitting and risk-threshold selection use disjoint source/duplicate components from the original calibration partition. The 327-crop assessment was inspected in the preceding experiment, so subsequent scores are post-test exploratory regression evidence.

The code cells preserve the archived execution configuration and dated controller limits. Recorded measurements and their scope are summarized below. Reproduction requires the recorded source and data identities; a later experiment uses a separate configuration and output directory.


## 1. Runtime and source identity

The initialization stage connects persistent storage, verifies the source archive and its manifest, records the Colab CUDA library pair and creates a versioned output directory.


In [ ]:
import os, sys, json, hashlib, zipfile, subprocess, uuid
from pathlib import Path, PurePosixPath
from importlib.metadata import version
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
from google.colab import drive, files
import torch
assert torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0), "Chọn GPU A100 trong Colab."
drive.mount("/content/drive")
SOURCE_SHA256 = "016b81f5f6ce595bba68513fe39b1b4f509563f6748dd174d02422b4d22b32ad"
ROOT = Path("/content/drive/MyDrive/G-M10_GarmentLab_ImproveToday")
ROOT.mkdir(parents=True, exist_ok=True)
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()
SOURCE = ROOT/"source"/SOURCE_SHA256/"GarmentLab_A100_ImproveToday_Source.zip"
SOURCE.parent.mkdir(parents=True, exist_ok=True)
if not SOURCE.is_file() or sha256_file(SOURCE) != SOURCE_SHA256:
    print("Chọn GarmentLab_A100_ImproveToday_Source.zip")
    uploaded = files.upload()
    matching = [name for name,value in uploaded.items() if hashlib.sha256(value).hexdigest() == SOURCE_SHA256]
    assert len(matching) == 1, "Source chưa khớp notebook. Chọn đúng ZIP ImproveToday đi kèm; giữ nguyên ZIP."
    if SOURCE.exists(): SOURCE.replace(SOURCE.with_name(SOURCE.name+".unverified_"+uuid.uuid4().hex[:8]))
    SOURCE.write_bytes(uploaded[matching[0]])
    del uploaded
assert sha256_file(SOURCE) == SOURCE_SHA256
REPO = Path("/content/GarmentLabImprove_"+SOURCE_SHA256[:12])
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE) as archive:
    manifest = json.loads(archive.read("SOURCE_MANIFEST.json"))
    entries = manifest["payloads"]
    paths = {item["path"] for item in entries}|{"SOURCE_MANIFEST.json"}
    assert len(paths)==len(archive.namelist()) and paths==set(archive.namelist())
    assert sum(item["bytes"] for item in entries) < 100*1024*1024
    for item in entries:
        path = PurePosixPath(item["path"])
        assert not path.is_absolute() and ".." not in path.parts and ":" not in str(path) and "\\" not in str(path)
        value = archive.read(str(path))
        assert len(value)==item["bytes"] and hashlib.sha256(value).hexdigest()==item["sha256"]
        target = REPO.joinpath(*path.parts)
        assert target.resolve().is_relative_to(REPO.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert sha256_file(target)==item["sha256"], "Source local khác bản đã kiểm chứng; dùng runtime sạch với cùng cặp tệp."
        else: target.write_bytes(value)
    (REPO/"SOURCE_MANIFEST.json").write_bytes(archive.read("SOURCE_MANIFEST.json"))
pair_before = {p:version(p) for p in ("torch","torchvision")}
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(REPO/"requirements-garmentlab-improve-colab.txt")],check=True)
assert pair_before=={p:version(p) for p in pair_before}, "Cặp CUDA đã thay đổi; dừng và giữ output để kiểm tra."
OLD_CACHE = Path("/content/drive/MyDrive/G-M10_GarmentLab_A100_Deadline/cache")
CACHE = OLD_CACHE if OLD_CACHE.is_dir() else ROOT/"cache"
CACHE.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(CACHE/"torch")
OUT = ROOT/"runs"/("run01_"+SOURCE_SHA256[:12])
WORK = Path("/content/garmentlab_improve_work_"+SOURCE_SHA256[:12])
command = [sys.executable,"-u",str(REPO/"tools/run_garmentlab_improve.py"),
           "--out",str(OUT),"--work",str(WORK),"--cache",str(CACHE)]
print("GPU:",torch.cuda.get_device_name(0))
print("CUDA PAIR:",pair_before)
print("OUTPUT:",OUT)
print("CACHE:",CACHE)


## 2. Study plan and software checks

The plan fixes candidate configurations and data roles. Synthetic CPU fixtures exercise transaction recovery and calibration; a full MobileNet forward/backward checks the A100 runtime. Synthetic scores assess software behavior rather than garment-classifier quality.


In [ ]:
subprocess.run(command+["--stage","plan"],cwd=REPO,check=True)
subprocess.run([sys.executable,"-m","pytest","-q",
    "tests/test_garmentlab_study_v2.py","tests/test_garmentlab_calibration_v2.py",
    "tests/test_garmentlab_evaluate_v2.py","--basetemp","/content/garmentlab_v2_smoke_"+SOURCE_SHA256[:12]],cwd=REPO,check=True)
subprocess.run(command+["--stage","check"],cwd=REPO,check=True)
CHECKS_OK = True
print("SOFTWARE / A100 CHECKS: OK")


## 3. Preparation, training and frozen evaluation

The controller reuses verified downloads, reconstructs the recorded arrays and data roles, and executes discovery, confirmation and paired final fits. Model and calibration choices are frozen before assessment inference. Committed transactions support continuation with the same configuration and run identity.


In [ ]:
assert CHECKS_OK
LOG = OUT/"notebook_console.log"
OUT.mkdir(parents=True,exist_ok=True)
def run_stage(stage):
    import signal
    with LOG.open("a",encoding="utf-8") as stream:
        process=subprocess.Popen(command+["--stage",stage],cwd=REPO,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end="",flush=True);stream.write(line);stream.flush()
            return process.wait()
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT)
            try: process.wait(timeout=45)
            except subprocess.TimeoutExpired:
                process.kill();process.wait(timeout=5)
            print("Đã dừng. Giữ nguyên Drive và source để resume; lease có thể cần khoảng 10 phút hết hạn.")
            raise
RUN_EXIT = run_stage("all")
STATUS_FILE=OUT/"RUN_STATUS.json"
print("RUN EXIT:",RUN_EXIT)
if STATUS_FILE.exists():
    current=json.loads(STATUS_FILE.read_text(encoding="utf-8"))
    print("CONTROLLER:",current.get("status"))
    if current.get("message"):print(current["message"])
if RUN_EXIT != 0:
    print("Lượt chạy chưa hoàn tất. ZIP ở bước sau là kết quả một phần hoặc bằng chứng lỗi, không phải kết quả tối ưu đầy đủ.")


## 4. Evidence export

The export stage packages available checkpoints, histories, frozen choices, logits and evaluation receipts. Study completion, technical service readiness, conditional risk-bound eligibility and classification quality are recorded separately.


In [ ]:
if LOG.exists():
    (OUT/"CONSOLE_CAPTURE.txt").write_bytes(LOG.read_bytes())
subprocess.run(command+["--stage","export"],cwd=REPO,check=True)
RESULT=OUT/"exports/g_m10_garmentlab_improve_today_run01_results.zip"
assert RESULT.is_file()
FINAL=OUT/"FINAL_SUMMARY_V2.json"
if FINAL.is_file():
    summary=json.loads(FINAL.read_text(encoding="utf-8"))
    print("RESEARCH:",summary.get("status"))
    print("LOCAL RELEASE READY:",summary.get("ready_for_local_service"))
    print("ASSESSMENT SCOPE:",summary.get("assessment_scope"))
    print("CALIBRATION POLICY:",summary.get("calibration",{}).get("policy"))
else:
    print("CHƯA CÓ FINAL SUMMARY: đây là ZIP một phần hoặc ZIP lỗi; giữ nguyên thư mục để tiếp tục.")
print("RESULT ZIP:",RESULT)
print("SHA256:",sha256_file(RESULT))
files.download(str(RESULT))


## Recorded A100 results · 28 October 2026

The exported run completed **20 fits × 18 epochs**. The original controller recorded approximately **89.67 active minutes**, including preparation and finalization; this is not total GPU allocation time. The code cells retain the original workflow; this publication copy has no execution outputs. Measurements in this section come from `FINAL_SUMMARY_V2.json` and the independent serving receipts.

C4 increases classifier dropout from 0.2 to 0.4. It won the declared mean selection Macro-F1 criterion on fresh confirmation seeds 202 and 303. Final seeds were 314, 2718 and 5772; deployment seed 314 was fixed before evaluation.

| Recipe | Final selection Macro-F1 | Reused assessment Macro-F1 |
|---|---:|---:|
| C0, dropout 0.2 | 0.84245 ± 0.00597 | 0.77472 ± 0.01493 |
| C4, dropout 0.4 | 0.84347 ± 0.00383 | 0.76638 ± 0.02258 |

Values are mean ± sample SD across three final training seeds. C4 does **not** demonstrate an assessment F1 gain. A paired component-bootstrap interval for the mean F1 difference C4–C0 is **[−0.03820, +0.01829]**, which includes zero. Assessment contains the same previously inspected 327 crops and is explicitly **post-test exploratory regression**, not a new blind test.

The preset C4 seed 314 scores **82.26% accuracy, Macro-F1 0.77553**. Disjoint temperature and threshold partitions give temperature **5.345757** and active review threshold **0.88**. The frozen policy accepts **182/327** assessment crops, with **173 correct**: **95.05% accepted accuracy at 55.66% coverage**. The component-representative calibration bound is conditional on assumptions; historical calibration reuse and unproven component independence prevent an unconditional future-accuracy guarantee.

The returned ONNX artifact passes the original 64-case protocol at the fixed 1e-4 logit tolerance. The broader Windows check on all 327 crops at batch 1 **fails ONNX** with maximum error **1.3542e-4**; three additional variants also fail. Original artifacts and tolerances remain unchanged. Local PyTorch CPU passes with maximum error **5.3406e-5**, without label or review-decision differences on those cases, and is the primary demo backend.

Seven PNG/Predictor checks and the in-process HTTP contracts pass. A separate 60-second local PyTorch v2 soak records **5,312/5,312 HTTP 200** predictions, p95 **31.43 ms**, at concurrency 2 on two repeated JPEG/crop cases. It does not establish independent-image quality, a hosted SLA or production reliability.

[Results, reproduction and local inference](../docs/garmentlab/REPRODUCIBILITY.md) · [AI usage](../AI_USAGE.md)
